# Spindle — Star Schema Transform

Spindle generates normalized 3NF data and can transform it into a **star schema** (dimension + fact tables) ready for Power BI DirectLake mode.

**What this notebook covers**
- Generating the Retail domain
- Transforming to a star schema with `StarSchemaTransform`
- Exploring dimension and fact tables
- The auto-generated `dim_date` table
- Surrogate keys (`sk_*`) and natural key preservation (`nk_*`)
- Writing the star schema to a Fabric Lakehouse
- Healthcare star schema

In [1]:
%pip install sqllocks-spindle --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\suref\OneDrive\VSCode\AzureClients\forge-workspace\projects\fabric-datagen\.venv-win\Scripts\python.exe -m pip install --upgrade pip


## Generate and transform — Retail

In [2]:
from sqllocks_spindle import Spindle, RetailDomain
from sqllocks_spindle.transform import StarSchemaTransform

spindle = Spindle()
result = spindle.generate(domain=RetailDomain(), scale="fabric_demo", seed=42)

transform = StarSchemaTransform()
star = transform.transform(result.tables, RetailDomain().star_schema_map())
print(star.summary())

Star Schema Result
DIMENSIONS:
  dim_customer                     200 rows  9 cols
  dim_product                      100 rows  10 cols
  dim_store                        150 rows  6 cols
  dim_promotion                    200 rows  7 cols
  dim_date                       1,352 rows  14 cols
FACTS:
  fact_sale                      2,500 rows  20 cols
  fact_return                      170 rows  15 cols


## Dimension tables

In [3]:
# dim_customer — surrogate key as first column
star.dimensions["dim_customer"].head()

,sk_customer,customer_id,first_name,last_name,email,gender,loyalty_tier,signup_date,is_active
0,1,1,Dillon,Macdonald,hyoung@example.org,F,Silver,2025-08-11 20:36:31.081033514,true
1,2,2,Andrew,Paul,brianna69@example.com,M,Basic,2024-11-18 05:11:09.502839751,true
2,3,3,Jasmine,Schaefer,lsmith@example.net,M,Platinum,2025-11-07 06:55:16.583412953,true
3,4,4,John,Evans,walvarez@example.net,M,Gold,2025-02-15 17:27:40.257596628,true
4,5,5,Andrea,Ramos,teresa60@example.org,M,Basic,2025-06-19 12:43:20.628112464,true


In [4]:
# dim_product — enriched with product_category columns (prefixed 'cat_')
star.dimensions["dim_product"].head()

,sk_product,product_id,category_id,product_name,unit_price,product_status,cost,cat_category_name,cat_parent_category_id,cat_level
0,1,1,7,Jump Rope Speed,19.48,active,9.46,Bedding & Bath,None,1
1,2,2,40,Dried Mango Slices,16.21,active,10.97,Garden & Outdoor,10,3
2,3,3,8,Almond Butter Natural,4.75,active,2.82,Women's Apparel,None,1
3,4,4,47,Noise-Cancelling Headphones,11.98,active,4.63,Toys & Games,20,3
4,5,5,6,Protein Bars Variety,50.13,active,32.27,Bedding & Bath,None,1


In [5]:
# Category columns added by enrichment
cat_cols = [c for c in star.dimensions["dim_product"].columns if c.startswith("cat_")]
print(f"Enriched category columns: {cat_cols}")

Enriched category columns: ['cat_category_name', 'cat_parent_category_id', 'cat_level']


## dim_date — auto-generated

`dim_date` is automatically generated from the date range found in your fact data.
The `sk_date` key uses YYYYMMDD integer format for efficient range filtering.

In [6]:
star.date_dim.head(10)

,sk_date,date,year,quarter,month,month_name,week_of_year,day_of_month,day_of_week,day_of_week_name,is_weekend,is_weekday,fiscal_year,fiscal_quarter
0,20220506,2022-05-06,2022,2,5,May,18,6,5,Friday,False,True,2022,2
1,20220507,2022-05-07,2022,2,5,May,18,7,6,Saturday,True,False,2022,2
2,20220508,2022-05-08,2022,2,5,May,18,8,7,Sunday,True,False,2022,2
3,20220509,2022-05-09,2022,2,5,May,19,9,1,Monday,False,True,2022,2
4,20220510,2022-05-10,2022,2,5,May,19,10,2,Tuesday,False,True,2022,2
5,20220511,2022-05-11,2022,2,5,May,19,11,3,Wednesday,False,True,2022,2
6,20220512,2022-05-12,2022,2,5,May,19,12,4,Thursday,False,True,2022,2
7,20220513,2022-05-13,2022,2,5,May,19,13,5,Friday,False,True,2022,2
8,20220514,2022-05-14,2022,2,5,May,19,14,6,Saturday,True,False,2022,2
9,20220515,2022-05-15,2022,2,5,May,19,15,7,Sunday,True,False,2022,2


In [7]:
print(f"Date range: {star.date_dim['date'].min()} → {star.date_dim['date'].max()}")
print(f"Total days: {len(star.date_dim):,}")
print(f"Columns: {list(star.date_dim.columns)}")

Date range: 2022-05-06 → 2026-01-16
Total days: 1,352
Columns: ['sk_date', 'date', 'year', 'quarter', 'month', 'month_name', 'week_of_year', 'day_of_month', 'day_of_week', 'day_of_week_name', 'is_weekend', 'is_weekday', 'fiscal_year', 'fiscal_quarter']


## Fact tables

Fact tables contain surrogate key (`sk_*`) columns for joining to dimensions, plus preserved natural keys (`nk_*`) for traceability.

In [8]:
star.facts["fact_sale"].head()

,order_line_id,order_id,quantity,unit_price,discount_percent,line_total,shipping_address_id,promotion_id_order,order_date,status,order_total,nk_customer_id,sk_customer,nk_product_id,sk_product,nk_store_id,sk_store,nk_promotion_id,sk_promotion,sk_date
0,1,45,8.0,19.48,0.0,155.84,None,None,2025-12-09 11:36:57.504110868,completed,231.35,12,12,1,1,5,5,None,<NA>,20251209
1,2,526,1.0,16.21,0.0,16.21,None,None,2022-04-26 19:47:58.000000000,completed,16.21,142,142,2,2,2,2,None,<NA>,20220426
2,3,817,1.0,11.98,0.0,11.98,None,None,2025-08-30 08:04:06.817461025,completed,35.94,75,75,4,4,4,4,None,<NA>,20250830
3,4,809,1.0,16.21,0.0,16.21,None,None,2024-11-03 07:41:03.000000000,completed,79.75,10,10,2,2,2,2,None,<NA>,20241103
4,5,716,1.0,10.82,0.0,10.82,197,None,2024-11-19 05:11:09.502839751,completed,163.89,2,2,21,21,1,1,None,<NA>,20241119


In [9]:
# sk_date is YYYYMMDD integer — joins directly to dim_date
sale = star.facts["fact_sale"]
print(f"sk_date sample: {sale['sk_date'].dropna().head().tolist()}")
print(f"SK columns:     {[c for c in sale.columns if c.startswith('sk_')]}")
print(f"NK columns:     {[c for c in sale.columns if c.startswith('nk_')]}")

sk_date sample: [20251209, 20220426, 20250830, 20241103, 20241119]
SK columns:     ['sk_customer', 'sk_product', 'sk_store', 'sk_promotion', 'sk_date']
NK columns:     ['nk_customer_id', 'nk_product_id', 'nk_store_id', 'nk_promotion_id']


## Referential integrity check

In [10]:
fact_sale = star.facts["fact_sale"]
dim_customer = star.dimensions["dim_customer"]
dim_product  = star.dimensions["dim_product"]

valid_customers = set(dim_customer["sk_customer"])
valid_products  = set(dim_product["sk_product"])

assert fact_sale["sk_customer"].dropna().isin(valid_customers).all(), "Customer SK mismatch"
assert fact_sale["sk_product"].dropna().isin(valid_products).all(),  "Product SK mismatch"
print("SK referential integrity: PASS")

SK referential integrity: PASS


## Write star schema to Fabric Lakehouse

> **Remove or skip this section when running locally.** Requires `spark` to be available (Fabric notebook).

In [11]:
# Uncomment to run inside a Fabric notebook
# for table_name, df in star.all_tables().items():
#     spark.createDataFrame(df).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"retail_star_{table_name}")
#     print(f"  retail_star_{table_name:<25} {len(df):>6,} rows")
print("Uncomment the block above to write to your Fabric Lakehouse.")

Uncomment the block above to write to your Fabric Lakehouse.


## Healthcare star schema

In [12]:
from sqllocks_spindle import HealthcareDomain

hc_result = spindle.generate(domain=HealthcareDomain(), scale="fabric_demo", seed=42)
hc_star = transform.transform(hc_result.tables, HealthcareDomain().star_schema_map())
print(hc_star.summary())

Star Schema Result
DIMENSIONS:
  dim_patient                      100 rows  18 cols
  dim_provider                      20 rows  8 cols
  dim_facility                      50 rows  12 cols
  dim_date                       1,384 rows  14 cols
FACTS:
  fact_encounter                   500 rows  11 cols
  fact_claim                     1,187 rows  18 cols

In [13]:
hc_star.facts["fact_encounter"].head()

,encounter_id,encounter_type,encounter_date,status,nk_patient_id,sk_patient,nk_provider_id,sk_provider,nk_facility_id,sk_facility,sk_date
0,1,Outpatient,2023-05-18 04:58:04.374324944,Completed,2,2,4,4,8,8,20230518
1,2,Emergency,2024-10-16 04:17:34.000000000,Completed,1,1,1,1,2,2,20241016
2,3,Outpatient,2024-12-20 18:49:15.411381257,Completed,13,13,8,8,1,1,20241220
3,4,Observation,2024-01-19 01:14:31.000000000,Completed,2,2,7,7,1,1,20240119
4,5,Outpatient,2024-07-16 10:17:27.182079513,Completed,1,1,3,3,3,3,20240716


## CDM export

Spindle can also export to Microsoft Common Data Model (CDM) format — compatible with Dataverse, Power Platform, and Fabric CDM connectors.

In [14]:
from sqllocks_spindle.transform import CdmMapper

mapper = CdmMapper()
entity_map = RetailDomain().cdm_map()

# In-memory model.json manifest
model = mapper.to_model_json(result.tables, domain_name="SpindleRetail", entity_map=entity_map)

print(f"CDM model name: {model['name']}")
print(f"Entities: {[e['name'] for e in model['entities']]}")

CDM model name: SpindleRetail
Entities: ['Contact', 'CustomerAddress', 'ProductCategory', 'Product', 'Campaign', 'Store', 'SalesOrder', 'SalesOrderProduct', 'ReturnOrder']


In [15]:
# Write CDM folder to disk (local or ADLS path)
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    files = mapper.write_cdm_folder(
        result.tables,
        output_dir=tmpdir,
        domain_name="SpindleRetail",
        entity_map=entity_map,
    )
    print(f"Written {len(files)} files:")
    for f in files:
        print(f"  {os.path.relpath(f, tmpdir)}")

Written 10 files:
  Contact\Contact.csv
  CustomerAddress\CustomerAddress.csv
  ProductCategory\ProductCategory.csv
  Product\Product.csv
  Campaign\Campaign.csv
  Store\Store.csv
  SalesOrder\SalesOrder.csv
  SalesOrderProduct\SalesOrderProduct.csv
  ReturnOrder\ReturnOrder.csv
  model.json
